# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,
    scroll_events
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
LIMIT 100
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,67,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,616,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,28,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,25,<NA>


In [5]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
""").df()


features["ctr"] = (
    features["gsc_clicks"] / features["gsc_impressions"]
).fillna(0)
features["score"] = 0

features.loc[
    (features["gsc_impressions"] >= 100) &
    (features["ctr"] < 0.02),
    "score"
] = 2

features.loc[
    (features["gsc_avg_position"] > 10),
    "score"
] += 1
features["reason_code"] = "NO_ACTION"

features.loc[
    (features["gsc_impressions"] > 100) &
    (features["ctr"] < 0.02),
    "reason_code"
] = "LOW_CTR_HIGH_IMPRESSIONS"

features.loc[
    (features["gsc_avg_position"] > 20),
    "reason_code"
] = "LOW_POSITION"
features["action"] = "No Action"

features.loc[
    features["reason_code"] == "LOW_CTR_HIGH_IMPRESSIONS",
    "action"
] = "Refresh Content"

features.loc[
    features["reason_code"] == "LOW_POSITION",
    "action"
] = "Improve SEO"


features.head(20)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0.000000,0,NO_ACTION,No Action
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.000000,0,NO_ACTION,No Action
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0.008000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0.000000,0,NO_ACTION,No Action
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0.000000,0,NO_ACTION,No Action
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,0.004184,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,0.000000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,0.000000,0,NO_ACTION,No Action
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,0.000000,0,NO_ACTION,No Action
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,0.000000,0,NO_ACTION,No Action


In [13]:
!git clone https://github.com/bibaibrahim168-svg/ML-assignment.git

Cloning into 'ML-assignment'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 157 (delta 67), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 1.88 MiB | 4.77 MiB/s, done.
Resolving deltas: 100% (67/67), done.


In [16]:
import pandas as pd

df = pd.read_csv(
    "ML-assignment/data/raw/content_refresh_anonymized.csv"
)

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print(df.shape)

(30000, 45)


In [21]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [22]:
model_features = [
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "word_count",
    "ctr",
    "days_since_last_update"
]

label = "is_declining_label"

X = df[model_features]
y = df[label]
groups = df["client_id"]

print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)

X: (30000, 6)
y: (30000,)
groups: (30000,)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1

The paper found leakage-related reproducibility problems across 17 fields, collectively affecting 329 papers.

**Methodology question:**

How were the papers selected for the survey, and does the validation of the survey support generalizing this finding to ML-based science broadly?

I would also ask how the reported reproducibility problem was defined and identified across the surveyed papers.

This is a constructive question because the paper's survey identifies reported reproducibility problems across research communities rather than establishing that every ML paper has the same issue.

### Finding 2

In the civil war prediction case study, the paper found that papers claiming that complex ML models outperform Logistic Regression failed to reproduce because of data leakage. After correcting the leakage, the complex ML models did not perform substantively better than Logistic Regression.

**Methodology question:**

Does the reproduction and validation design use an honest and comparable evaluation setup after correcting the leakage, so that the before-and-after performance comparison supports this claim?

I would also ask how the prediction label was defined and whether the same label definition and evaluation metric were preserved when comparing the models.

This question focuses on whether the evaluation setup fairly represents the prediction task rather than questioning the authors' conclusion.

### Overall note

These questions are intended to strengthen the interpretation of the findings by checking the label definition, leakage risks, and validation design before making broader claims.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


I re-ran the Week-5 Random Forest using a grouped-by-client split.

The grouping was done by `client_id`, so records from the same client were kept together and were not split between the training and test sets.

I kept the same six model features, the same Random Forest method, and the same Precision@50 metric so that the validation design was the main change.

### Before

Week-5 Random Forest Precision@50 on the original split: 0.600

### After

Grouped-by-client Random Forest Precision@50: 0.540

The grouped split used 23,837 training rows from 25 clients and 6,163 test rows from 7 different clients.

### Interpretation

The grouped-by-client split is a more honest validation design for this question because the model is evaluated on clients that were not represented in the training data.

The measured Precision@50 under the grouped split was 0.540. The measured performance was lower than the Week-5 original-split result, showing that the validation design affects the measured model performance.

The Hand Rule baseline measured Precision@50 of 0.720. This is a separate baseline result and is not the original Random Forest result.

These results are directional decision-support evidence for the evaluated data and validation design. They do not establish that the model will always perform the same way on new or future clients.

In [23]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Unique train clients:", groups.iloc[train_idx].nunique())
print("Unique test clients :", groups.iloc[test_idx].nunique())

Train: (23837, 6)
Test : (6163, 6)
Unique train clients: 25
Unique test clients : 7


In [27]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    max_depth=6,
    random_state=42
)

In [28]:
model.fit(X_train, y_train)

test_scores = model.predict_proba(X_test)[:, 1]

In [29]:
results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "score": test_scores
})

top_50 = results.sort_values(
    "score",
    ascending=False
).head(50)

grouped_precision_at_50 = top_50["actual"].mean()

print(
    f"Grouped-client Precision@50: "
    f"{grouped_precision_at_50:.3f}"
)

Grouped-client Precision@50: 0.540


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I audited the final feature set for possible target leakage, identifier leakage, and future-information risks.

The model uses these six features:

- impressions_90d
- avg_position
- content_age_days
- word_count
- ctr
- days_since_last_update

### Leakage checks

1. **Target leakage**

The target is `is_declining_label`, which is derived from `trend_direction`.

The target column and `trend_direction` were not included as model features.

2. **Identifier leakage**

`content_id` and `client_id` were not used as model features.

3. **Future-information risk**

The feature definitions were reviewed for whether they could contain information that would only be available after the prediction point.

The audit did not identify an obvious future-information feature from the final six-feature list, but this review does not prove that all possible temporal leakage has been eliminated.

4. **Feature availability**

The final feature set was reviewed against the intended prediction setting to check that the features represent information that could reasonably be available when making the prediction.

### Conclusion

No direct target leakage was identified in the final six-feature set based on these checks.

The audit reduces obvious leakage risks, but it does not prove that every possible source of leakage has been eliminated.

In [35]:
print("Target:", label)
print("Model features:", model_features)

print(
    "Target included in features:",
    label in model_features
)

print(
    "Trend direction included in features:",
    "trend_direction" in model_features
)

print(
    "content_id included:",
    "content_id" in model_features
)

print(
    "client_id included:",
    "client_id" in model_features
)

print(df[model_features].head())
print(df[model_features].isna().sum())

Target: is_declining_label
Model features: ['impressions_90d', 'avg_position', 'content_age_days', 'word_count', 'ctr', 'days_since_last_update']
Target included in features: False
Trend direction included in features: False
content_id included: False
client_id included: False
   impressions_90d  avg_position  content_age_days  word_count   ctr  \
0             3803          10.6               187      3221.0  0.76   
1            15320          20.3               445      2481.0  0.05   
2            12581          36.5               141      3515.0  0.09   
3            11751           6.2               463         NaN  0.49   
4            19140          44.0               263      2803.0  0.13   

   days_since_last_update  
0                      20  
1                      25  
2                      20  
3                      22  
4                      14  
impressions_90d              0
avg_position                 0
content_age_days             0
word_count                76

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Random Forest model is better than the hand-written rule.

### Safer claim

The hand-written rule measured Precision@50 of 0.720 on the evaluated baseline data, while the Random Forest measured Precision@50 of 0.540 under the grouped-by-client split.

Under this validation setup, the measured result did not show an advantage for the Random Forest model over the hand-written baseline.

This result is directional decision-support evidence for the evaluated data and validation design. It does not establish future performance on new clients.

In [36]:
print(f"Grouped-client Precision@50: {grouped_precision_at_50:.3f}")
print("Hand Rule Precision@50: 0.720")

Grouped-client Precision@50: 0.540
Hand Rule Precision@50: 0.720


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.